In [2]:
import os

# 指定只对当前进程可见的显卡
# 注意：这里设置为 "1" 后，程序内部看到的 cuda:0 就是这张卡
os.environ["CUDA_VISIBLE_DEVICES"] = "3"

import time
import numpy as np
import scipy.io as sio
import torch
import triton
import triton.language as tl


# ==========================================
# 0. 路径与文件配置
# ==========================================
input_dir = "./"
output_dir = "./"

signal_files = [
    "data/processed_signal_group03_pose000.txt",
    "data/processed_signal_group03_pose001.txt",
    "data/processed_signal_group03_pose002.txt",
    "data/processed_signal_group03_pose003.txt",
    "data/processed_signal_group03_pose004.txt",
    "data/processed_signal_group03_pose005.txt",
    "data/processed_signal_group03_pose006.txt",
    "data/processed_signal_group03_pose007.txt",
    "data/processed_signal_group03_pose008.txt",
    "data/processed_signal_group03_pose009.txt",
    # "data/processed_signal_group03_pose010.txt",
    # "data/processed_signal_group03_pose011.txt",
    # "data/processed_signal_group03_pose012.txt",
    # "data/processed_signal_group03_pose013.txt",
    # "data/processed_signal_group03_pose014.txt",
    # "data/processed_signal_group03_pose015.txt",
    # "data/processed_signal_group03_pose016.txt",
    # "data/processed_signal_group03_pose017.txt",
    # "data/processed_signal_group03_pose018.txt",
    # "data/processed_signal_group03_pose019.txt",
    # "data/processed_signal_group03_pose020.txt",
    # "data/processed_signal_group03_pose021.txt",
    # "data/processed_signal_group03_pose022.txt",
    # "data/processed_signal_group03_pose023.txt",
    # "data/processed_signal_group03_pose024.txt",
    # "data/processed_signal_group03_pose025.txt",
    # "data/processed_signal_group03_pose026.txt",
    # "data/processed_signal_group03_pose027.txt",
    # "data/processed_signal_group03_pose028.txt",
]

location_files = [
    "data/sensor_location_group03_pose000.txt",
    "predicted_locations_group3_pose1_10MHz_baseon0_corrected_finetuned_masked_10MHz.txt",
    "transformed_locations_group3_pose2_to_pose_of_0.txt",
    "transformed_locations_group3_pose3_to_pose_of_0.txt",
    "transformed_locations_group3_pose4_to_pose_of_0.txt",
    "transformed_locations_group3_pose5_to_pose_of_0.txt",
    "transformed_locations_group3_pose6_to_pose_of_0.txt",
    "transformed_locations_group3_pose7_to_pose_of_0.txt",
    "transformed_locations_group3_pose8_to_pose_of_0.txt",
    "transformed_locations_group3_pose9_to_pose_of_0.txt",
    # "transformed_locations_group3_pose10_to_pose_of_0.txt",
    # "transformed_locations_group3_pose11_to_pose_of_0.txt",
    # "transformed_locations_group3_pose12_to_pose_of_0.txt",
    # "transformed_locations_group3_pose13_to_pose_of_0.txt",
    # "transformed_locations_group3_pose14_to_pose_of_0.txt",
    # "transformed_locations_group3_pose15_to_pose_of_0.txt",
    # "transformed_locations_group3_pose16_to_pose_of_0.txt",
    # "transformed_locations_group3_pose17_to_pose_of_0.txt",
    # "transformed_locations_group3_pose18_to_pose_of_0.txt",
    # "transformed_locations_group3_pose19_to_pose_of_0.txt",
    # "transformed_locations_group3_pose20_to_pose_of_0.txt",
    # "transformed_locations_group3_pose21_to_pose_of_0.txt",
    # "transformed_locations_group3_pose22_to_pose_of_0.txt",
    # "transformed_locations_group3_pose23_to_pose_of_0.txt",
    # "transformed_locations_group3_pose24_to_pose_of_0.txt",
    # "transformed_locations_group3_pose25_to_pose_of_0.txt",
    # "transformed_locations_group3_pose26_to_pose_of_0.txt",
    # "transformed_locations_group3_pose27_to_pose_of_0.txt",
    # "transformed_locations_group3_pose28_to_pose_of_0.txt",
]

assert len(signal_files) == len(location_files), "错误：信号文件名列表和位置文件名列表的数量不一致！"

output_file_prefix = "compensate_width30_3Dpano_from_txt_group0_2_9_loc_pred"


# ==========================================
# 1. 算法选择与参数配置
# ==========================================
recon_method = "DAS"  # 可选: "DAS" 或 "FBP"

vs = 1500.0
fs = 40e6
num_times = 4096
res = 0.1e-3

x_start, x_end = -12.5e-3, 17.5e-3
y_start, y_end = -10.0e-3, 40.0e-3
z_start, z_end = -20.0e-3, 0.0e-3

num_x = np.around((x_end - x_start) / res).astype(np.int32)
num_y = np.around((y_end - y_start) / res).astype(np.int32)
num_z = np.around((z_end - z_start) / res).astype(np.int32)

num_voxels = int(num_x) * int(num_y) * int(num_z)


# ==========================================
# 2. Triton Kernels
# ==========================================
@triton.jit
def recon_kernel_triton(
    signal_ptr,
    location_ptr,
    output_ptr,
    x_start: tl.constexpr,
    y_start: tl.constexpr,
    z_start: tl.constexpr,
    res: tl.constexpr,
    vs: tl.constexpr,
    fs: tl.constexpr,
    NUM_X: tl.constexpr,
    NUM_Y: tl.constexpr,
    NUM_Z: tl.constexpr,
    NUM_DETECTORS: tl.constexpr,
    NUM_TIMES: tl.constexpr,
    METHOD: tl.constexpr,       # 0 = DAS, 1 = FBP
    BLOCK_VOXELS: tl.constexpr,
    BLOCK_DETECTORS: tl.constexpr,
):
    pid = tl.program_id(0)

    voxel_offsets = pid * BLOCK_VOXELS + tl.arange(0, BLOCK_VOXELS)
    voxel_mask = voxel_offsets < (NUM_X * NUM_Y * NUM_Z)

    # output flatten 顺序对应 numpy C-order: [i, j, k]
    k = voxel_offsets % NUM_Z
    j = (voxel_offsets // NUM_Z) % NUM_Y
    i = voxel_offsets // (NUM_Y * NUM_Z)

    x = x_start + i.to(tl.float32) * res
    y = y_start + j.to(tl.float32) * res
    z = z_start + k.to(tl.float32) * res

    acc = tl.zeros((BLOCK_VOXELS,), dtype=tl.float32)

    det_block = tl.arange(0, BLOCK_DETECTORS)

    for det_start in range(0, NUM_DETECTORS, BLOCK_DETECTORS):
        det_idx = det_start + det_block
        det_mask = det_idx < NUM_DETECTORS

        loc_base = det_idx * 3

        det_x = tl.load(location_ptr + loc_base + 0, mask=det_mask, other=0.0)
        det_y = tl.load(location_ptr + loc_base + 1, mask=det_mask, other=0.0)
        det_z = tl.load(location_ptr + loc_base + 2, mask=det_mask, other=0.0)

        dx = x[:, None] - det_x[None, :]
        dy = y[:, None] - det_y[None, :]
        dz = z[:, None] - det_z[None, :]

        d2 = dx * dx + dy * dy + dz * dz
        d = tl.sqrt(d2)

        vector_n0 = tl.sqrt(det_x * det_x + det_y * det_y + det_z * det_z)

        dot_val = -dx * det_x[None, :] - dy * det_y[None, :] - dz * det_z[None, :]
        denom = vector_n0[None, :] * d

        angle_cos = dot_val / denom
        angle_cos = tl.maximum(angle_cos, 0.0)

        idx_float = d / vs * fs
        idx = idx_float.to(tl.int32)

        if METHOD == 0:
            valid = (
                voxel_mask[:, None]
                & det_mask[None, :]
                & (idx >= 0)
                & (idx < NUM_TIMES)
                & (d2 > 0.0)
                & (denom > 0.0)
            )

            signal_val = tl.load(
                signal_ptr + det_idx[None, :] * NUM_TIMES + idx,
                mask=valid,
                other=0.0,
            )

            contrib = signal_val * angle_cos / d2

        else:
            valid = (
                voxel_mask[:, None]
                & det_mask[None, :]
                & (idx >= 0)
                & (idx < NUM_TIMES - 2)
                & (d2 > 0.0)
                & (denom > 0.0)
            )

            signal_0 = tl.load(
                signal_ptr + det_idx[None, :] * NUM_TIMES + idx,
                mask=valid,
                other=0.0,
            )

            signal_1 = tl.load(
                signal_ptr + det_idx[None, :] * NUM_TIMES + idx + 1,
                mask=valid,
                other=0.0,
            )

            derivative = signal_1 - signal_0

            # 保持与你原 Taichi FBP 代码一致：
            # signal_backproj[n, idx] - idx_float * derivative
            contrib = (signal_0 - idx_float * derivative) * angle_cos / d2

        contrib = tl.where(valid, contrib, 0.0)
        acc += tl.sum(contrib, axis=1)

    tl.store(output_ptr + voxel_offsets, acc, mask=voxel_mask)


# ==========================================
# 3. 读取并拼接多组 txt 数据
# ==========================================
print(f"Loading and fusing {len(signal_files)} data pairs...")
start_load = time.time()

all_signals = []
all_locations = []

signal_scale_files = {
    "data/processed_signal_group03_pose000.txt": 3.0,
    "data/processed_signal_group03_pose009.txt": 3.0,
}

for sig_file, loc_file in zip(signal_files, location_files):
    signal_txt_path = os.path.join(input_dir, sig_file)
    location_txt_path = os.path.join(input_dir, loc_file)

    temp_signal = np.loadtxt(signal_txt_path, dtype=np.float32)
    temp_location = np.loadtxt(location_txt_path, dtype=np.float32)

    if sig_file in signal_scale_files:
        temp_signal *= signal_scale_files[sig_file]
        print(f"  - Scaled signal by {signal_scale_files[sig_file]}: {sig_file}")

    if temp_signal.ndim == 1:
        temp_signal = temp_signal[None, :]

    if temp_location.ndim == 1:
        temp_location = temp_location[None, :]

    if temp_signal.shape[1] != num_times:
        raise ValueError(
            f"{sig_file} 的时间采样点数为 {temp_signal.shape[1]}，但 num_times={num_times}"
        )

    if temp_location.shape[1] != 3:
        raise ValueError(
            f"{loc_file} 的位置列数为 {temp_location.shape[1]}，应为 3 列 x/y/z"
        )

    if temp_signal.shape[0] != temp_location.shape[0]:
        raise ValueError(
            f"{sig_file} 和 {loc_file} 的探测器数量不一致："
            f"{temp_signal.shape[0]} vs {temp_location.shape[0]}"
        )

    all_signals.append(temp_signal)
    all_locations.append(temp_location)

    print(f"  - Loaded: {sig_file} (Detectors: {temp_location.shape[0]})")

real_signal = np.ascontiguousarray(np.vstack(all_signals), dtype=np.float32)
sensor_location = np.ascontiguousarray(np.vstack(all_locations), dtype=np.float32)

num_detectors = int(sensor_location.shape[0])

print(f"Data loading and fusion finished in {time.time() - start_load:.2f} seconds.")
print(
    f"==> Equivalent DENSE Array Detectors: {num_detectors}, "
    f"Fused Signal shape: {real_signal.shape}"
)


# ==========================================
# 4. 拷贝到 GPU
# ==========================================
if not torch.cuda.is_available():
    raise RuntimeError("当前环境没有可用 CUDA GPU，Triton 版本需要 CUDA。")

device = torch.device("cuda")

signal_gpu = torch.from_numpy(real_signal).to(device=device, dtype=torch.float32).contiguous()
location_gpu = torch.from_numpy(sensor_location).to(device=device, dtype=torch.float32).contiguous()

output_gpu = torch.empty((num_voxels,), device=device, dtype=torch.float32)


# ==========================================
# 5. 执行 Triton 重建
# ==========================================
if recon_method == "DAS":
    method_id = 0
elif recon_method == "FBP":
    method_id = 1
else:
    raise ValueError("Invalid recon_method. Please choose 'DAS' or 'FBP'.")

print(
    f"Starting [{recon_method}] reconstruction on Grid: "
    f"{num_x}x{num_y}x{num_z} using Triton CUDA..."
)

start_recon = time.time()

# 可根据显存和速度调节：
# BLOCK_VOXELS 越大，单个 program 处理体素越多
# BLOCK_DETECTORS 越大，单次并行累加探测器越多，但显存/寄存器压力更高
BLOCK_VOXELS = 128
BLOCK_DETECTORS = 32

grid = (triton.cdiv(num_voxels, BLOCK_VOXELS),)

recon_kernel_triton[grid](
    signal_gpu,
    location_gpu,
    output_gpu,
    float(x_start),
    float(y_start),
    float(z_start),
    float(res),
    float(vs),
    float(fs),
    int(num_x),
    int(num_y),
    int(num_z),
    int(num_detectors),
    int(num_times),
    int(method_id),
    BLOCK_VOXELS=BLOCK_VOXELS,
    BLOCK_DETECTORS=BLOCK_DETECTORS,
    num_warps=4,
)

torch.cuda.synchronize()

end_recon = time.time()
print("Reconstruction time: {:.2f}s".format(end_recon - start_recon))


# ==========================================
# 6. 保存结果
# ==========================================
signal_recon = output_gpu.detach().cpu().numpy().reshape(
    int(num_x), int(num_y), int(num_z)
)

signal_recon_abs = np.abs(signal_recon).astype(np.float32)

num_fused = len(signal_files)
save_filename = f"{output_file_prefix}_{recon_method}_fused_{num_fused}files_triton.mat"
save_path = os.path.join(output_dir, save_filename)

os.makedirs(output_dir, exist_ok=True)

sio.savemat(save_path, {"signal_recon": signal_recon_abs})

print(f"Successfully saved reconstructed volume to {save_path}")

Loading and fusing 10 data pairs...
  - Scaled signal by 3.0: data/processed_signal_group03_pose000.txt
  - Loaded: data/processed_signal_group03_pose000.txt (Detectors: 1024)
  - Loaded: data/processed_signal_group03_pose001.txt (Detectors: 1024)
  - Loaded: data/processed_signal_group03_pose002.txt (Detectors: 1024)
  - Loaded: data/processed_signal_group03_pose003.txt (Detectors: 1024)
  - Loaded: data/processed_signal_group03_pose004.txt (Detectors: 1024)
  - Loaded: data/processed_signal_group03_pose005.txt (Detectors: 1024)
  - Loaded: data/processed_signal_group03_pose006.txt (Detectors: 1024)
  - Loaded: data/processed_signal_group03_pose007.txt (Detectors: 1024)
  - Loaded: data/processed_signal_group03_pose008.txt (Detectors: 1024)
  - Scaled signal by 3.0: data/processed_signal_group03_pose009.txt
  - Loaded: data/processed_signal_group03_pose009.txt (Detectors: 1024)
Data loading and fusion finished in 3.47 seconds.
==> Equivalent DENSE Array Detectors: 10240, Fused Signal 